# 01 Pull and Clean Data

Pulls raw data (SQL query or a file), removes blacklisted items, cleans the
description with a named cleaning method and writes everything to `data/<timestamp>/`:

| file | contents |
|---|---|
| `raw_data_<ts>.txt` | the raw pull after the blacklist filter |
| `cleaned_data_<ts>.txt` | rows kept, with the cleaned description in `CLEANED_COL` |
| `cleaner.pkl` | the compiled cleaning recipe (re-used by notebooks 02/03) |
| `vocabulary.txt` | words kept by step 11 (learned from this data; frozen into `cleaner.pkl`) |
| `<cleaning method>/` | copy of the cleaning-method folder used (code, lists, settings) |
| `metadata_<ts>.json` | source, filters, cleaning configuration, row counts, n-gram counts |

Notebooks 02 and 03 can use this folder directly (`PREPARED_DATA = "<timestamp>"`) or repeat
the same steps themselves in a single cell.

In [0]:
# ---- MLflow -------------------------------------------------------------------
USE_MLFLOW = False                    # False = no tracking at all
MLFLOW_EXPERIMENT = None              # None = this notebook's experiment; or e.g. "/Users/patrick.white@vallen.com/nb-classifier"

# ---- data source (RAW_DATA_FILE supersedes SQL_QUERY) -----------------------
RAW_DATA_FILE = None                  # e.g. "data/my_extract.txt"; read with dtype=str
RAW_DATA_DELIMITER = "|"              # delimiter of RAW_DATA_FILE
RAW_DATA_QUOTED = True                # False if RAW_DATA_FILE uses no quoting and descriptions may start with "
SQL_QUERY = "labeled_items"          # a file in data/queries/ (name without .sql); may be None only if RAW_DATA_FILE is set

# ---- items to exclude ---------------------------------------------------------
BLACKLIST_FILE = None                 # e.g. "data/blacklists/blacklist.txt" (one column)
BLACKLIST_COL = "Vallen ID"           # data column the blacklist values are matched against
BLACKLIST_HAS_HEADER = True           # first line of the blacklist file is a header

# ---- cleaning -----------------------------------------------------------------
CLEANING_METHOD = "cleaning_method_20261015"   # a folder in cleaning/ (code, term lists, settings in method.yaml)
APPEND_FIELDS = {}                    # e.g. {"Global Vendno": "VEND", "Prodline": "PLINE"} -> VEND_<value> tokens
APPEND_NULLS_AS_NONE = True           # True: null field -> VEND_None;  False: no marker for that row

# ---- columns ------------------------------------------------------------------
TEXT_COL = "FullDesc"                 # raw description field to clean
CLEANED_COL = "CleanDesc"             # header for the cleaned description
LEVEL_COLUMNS = ["Level 1 ID", "Level 2 ID", "Level 3 ID", "Level 4 ID"]  # rows missing a label are dropped; [] for unlabelled data
ID_COL = "ID"                         # used only to report duplicate identifiers

# ---- output -------------------------------------------------------------------
OUTPUT_DELIMITER = "|"                # delimiter of the .txt files written

In [0]:
# ---- setup (no settings below this line need changing) ----------------------
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import pandas as pd
from nbclassifier import cleaning, data, paths
from nbclassifier.tracking import Tracker

TS = paths.timestamp()
OUT_DIR = paths.DATA_DIR / TS
tracker = Tracker(USE_MLFLOW, MLFLOW_EXPERIMENT, run_name=f"01_pull_and_clean_{TS}",
                  tags={"notebook": "01_pull_and_clean_data"})
print("output folder:", OUT_DIR)

output folder: /Workspace/Users/patrick.white@vallen.com/Category Classification/New NB Repo/data/20261016_220856


## Pull raw data

In [0]:
raw, source = data.load_raw(RAW_DATA_FILE, SQL_QUERY, delimiter=RAW_DATA_DELIMITER, quoted=RAW_DATA_QUOTED,
                            spark=globals().get("spark"))
print(f"source : {source['type']} ({source.get('query_name') or source.get('path')})")
print(f"rows   : {len(raw):,}")
print(f"columns: {list(raw.columns)}")
n_dup = data.duplicate_count(raw, ID_COL)
if n_dup:
    print(f"WARNING: {n_dup:,} rows share an {ID_COL!r} with another row")
raw.head()

source : sql (labeled_items)
rows   : 814,963
columns: ['ID', 'Cono', 'Vallen ID', 'FullDesc', 'category_l4_id', 'Global Vendno', 'ICSP Prodcat', 'UNSPSC', 'Prodline', 'Level 1 ID', 'Level 1 Name', 'Level 2 ID', 'Level 2 Name', 'Level 3 ID', 'Level 3 Name', 'Level 4 ID', 'Level 4 Name']


,ID,Cono,Vallen ID,FullDesc,category_l4_id,Global Vendno,ICSP Prodcat,UNSPSC,Prodline,Level 1 ID,Level 1 Name,Level 2 ID,Level 2 Name,Level 3 ID,Level 3 Name,Level 4 ID,Level 4 Name
0,I8IGTHF0H3HT,1,00892502046,D0748 F BLADE 7-1/4X48TH FERROVS,L4-12810826,FREUD,2000,,1_FREUD_888,L1-12810533,Tools & Equipment,L2-12810536,Power Tools,L3-12810584,Power Saw Blades & Accessories,L4-12810826,Circular Saw Blades
1,IF3JYMND0HXW,1,00892538579,50-101 ROUTER BIT,L4-12784351,FREUD,500,,1_FREUD_1,L1-12783695,Cutting Tools,L2-12783697,Holemaking,L3-12783722,Specialty Drill Bits,L4-12784351,Router Drill Bits
2,I8HHYVD9KP78,1,008925DS0912BW5,9IN DEMO DEMON F/METAL,L4-12810833,FREUD,2000,,1_FREUD_1,L1-12810533,Tools & Equipment,L2-12810536,Power Tools,L3-12810584,Power Saw Blades & Accessories,L4-12810833,Reciprocating Saw Blades
3,I8HJI1MPX8DQ,1,01011981-34-34,81-34-34 MAGNIFIER 10X ILLUM CODDINGTON B&L,L4-12810909,BAUS,9900,,1_BAUS_999,L1-12810533,Tools & Equipment,L2-12810537,Test Instruments,L3-13340590,Inspection Equipment,L4-12810909,Magnifiers
4,IACF1JY0BD2N,1,01058203068,BLOCK GABLE 03068 1/4 CA BLE,L4-13322418,1-10582,1300,,1_1-10582_1,L1-12800755,Lifting & Rigging,L2-12800757,Chains & Ropes,L3-12800775,Ropes,L4-13322418,Ropes


## Blacklist

In [0]:
n_raw = len(raw)
n_black = 0
if BLACKLIST_FILE is not None:
    items = data.read_blacklist(BLACKLIST_FILE, has_header=BLACKLIST_HAS_HEADER)
    raw, n_black = data.apply_blacklist(raw, items, BLACKLIST_COL)
    print(f"{len(items):,} blacklisted items; {n_black:,} rows removed; {len(raw):,} remain")
else:
    print("no blacklist")

n_nolabel = 0
if LEVEL_COLUMNS:
    raw, n_nolabel = data.drop_missing_labels(raw, LEVEL_COLUMNS)
    print(f"{n_nolabel:,} rows dropped for a missing label in {LEVEL_COLUMNS}")

no blacklist
0 rows dropped for a missing label in ['Level 1 ID', 'Level 2 ID', 'Level 3 ID', 'Level 4 ID']


## Save raw data

In [0]:
OUT_DIR.mkdir(parents=True, exist_ok=False)
raw_path = OUT_DIR / f"raw_data_{TS}.txt"
raw.to_csv(raw_path, sep=OUTPUT_DELIMITER, index=False)
print(f"wrote {raw_path} ({len(raw):,} rows)")

wrote /Workspace/Users/patrick.white@vallen.com/Category Classification/New NB Repo/data/20261016_220856/raw_data_20261016_220856.txt (814,963 rows)


## Clean

Runs the 13 steps of `CLEANING_METHOD` (see `cleaning/<method>/method.yaml`), printing how many
descriptions each step changed with examples. Step 11's vocabulary is learned from this data.
Rows whose description is blank (before or after cleaning) are dropped, and marker tokens for
`APPEND_FIELDS` are added.

In [0]:
print("available cleaning methods:", cleaning.available_methods())
cleaner = cleaning.build_cleaner(CLEANING_METHOD, append_fields=APPEND_FIELDS,
                                 append_nulls_as_none=APPEND_NULLS_AS_NONE)
cleaned, counts = data.clean_dataframe(raw, cleaner, TEXT_COL, CLEANED_COL, drop_empty=True,
                                       fit=True, verbose=False)
print()
for k in ("rows_in", "blank_raw_dropped", "emptied_by_cleaning_dropped", "rows_out"):
    print(f"{k:<30}{counts[k]:>12,}")

# before / after examples
sample = cleaned.sample(min(15, len(cleaned)), random_state=0)[[TEXT_COL, CLEANED_COL]]
with pd.option_context("display.max_colwidth", 120):
    display(sample)

available cleaning methods: ['cleaning_method_20261015']
Cleaner(method='cleaning_method_20261015', english_words=234,377, vocabulary=not learned, append_fields={})
  lists: 92 units, 67 colors and abbreviations, 102 sizes, 590 good tokens

rows_in                            814,963
blank_raw_dropped                        0
emptied_by_cleaning_dropped          2,867
rows_out                           812,096


FullDesc,CleanDesc
(DV) SAW CIRCULAR 7-1/4 M18 TOOL ONLY,SAW CIRCULAR M18 TOOL ONLY
400028 PUNCH ADAPTOR RAM,PUNCH ADAPTOR RAM
MDW04219GS2 DRILL 27/64 2XD,DRILL 2XD
L0232-098 LUBRICANT FGL-2,LUBRICANT
5887064 1/2-13 H3 3F RHSF SBOTT HSS-E,H3 3F RHSF SBOTT HSSE
A4R0500M05P00GUP KCU25 INSERT 5147213,KCU25 INSERT
0304502 WHEEL FELT KNIFE EDGE 3/4 IN,WHEEL FELT KNIFE EDGE IN
BLADE SHOVEL SNOW POLYET HYLENE,BLADE SHOVEL SNOW POLYETHYLENE
DNMG431QMH13A INSERT T-MAX P TURNING,INSERT TMAX P TURNING
38942 COVERALL A35 3X HOODED ZIP ELAS WR&ANK,COVERALL A35 HOODED ZIP ELAS WRANK


In [0]:
# most frequent words after cleaning (sanity check of the cleaning method)
cleaning.top_tokens(cleaned[CLEANED_COL], n=30).T

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29
token,<SIZE>,<SIZE>IN,<COLOR>,DRILL,<SIZE>MM,GLOVE,BIT,HEX,HSS,<SIZE>FT,<SIZE>FL,INSERT,<SIZE>PT,SOCKET,SCREW,ENDMILL,FR,<SIZE>DR,TAP,<SIZE>V,HD,<SIZE>A,<SIZE>OZ,HOSE,SS,CUT,WRENCH,CAP,JOBBER,DISC
count,298242,151368,114670,47210,40196,27129,24671,23390,23008,21426,20735,20487,20282,19534,16963,15782,15541,15223,14776,14211,13612,13177,12976,12928,12427,12419,12199,11649,10800,10769


## Save cleaned data

In [0]:
row_counts = {
    "raw_rows": n_raw,
    "blacklisted_dropped": n_black,
    "missing_label_dropped": n_nolabel,
    "blank_raw_dropped": counts["blank_raw_dropped"],
    "emptied_by_cleaning_dropped": counts["emptied_by_cleaning_dropped"],
    "cleaned_rows": len(cleaned),
}
row_counts["total_dropped"] = row_counts["raw_rows"] - row_counts["cleaned_rows"]
stats = cleaning.ngram_stats(cleaned[CLEANED_COL])

metadata = {
    "created": TS,
    "source": source,
    "blacklist_file": str(paths.resolve(BLACKLIST_FILE)) if BLACKLIST_FILE else None,
    "blacklist_col": BLACKLIST_COL,
    "text_col": TEXT_COL,
    "cleaned_col": CLEANED_COL,
    "level_columns": LEVEL_COLUMNS or None,
    "cleaning": cleaner.describe(),
    "row_counts": row_counts,
    "ngram_stats": stats,
}
files = data.save_dataset(OUT_DIR, TS, raw=None, cleaned=cleaned, cleaner=cleaner,
                          metadata=metadata, delimiter=OUTPUT_DELIMITER)
for k, v in files.items():
    print(f"{k:<10} {v}")
print(f"\nUse in notebooks 02/03 with:  PREPARED_DATA = \"{TS}\"")

raw        /Workspace/Users/patrick.white@vallen.com/Category Classification/New NB Repo/data/20261016_220856/raw_data_20261016_220856.txt
cleaned    /Workspace/Users/patrick.white@vallen.com/Category Classification/New NB Repo/data/20261016_220856/cleaned_data_20261016_220856.txt
cleaner    /Workspace/Users/patrick.white@vallen.com/Category Classification/New NB Repo/data/20261016_220856/cleaner.pkl
cleaning_method /Workspace/Users/patrick.white@vallen.com/Category Classification/New NB Repo/data/20261016_220856/cleaning_method_20261015
vocabulary /Workspace/Users/patrick.white@vallen.com/Category Classification/New NB Repo/data/20261016_220856/vocabulary.txt
metadata   /Workspace/Users/patrick.white@vallen.com/Category Classification/New NB Repo/data/20261016_220856/metadata_20261016_220856.json

Use in notebooks 02/03 with:  PREPARED_DATA = "20261016_220856"


## Vocabulary size

In [0]:
data.print_summary(row_counts, stats)

tracker.log_params({"source": {k: v for k, v in source.items() if k != "query_text"},
                    "cleaning_method": CLEANING_METHOD,
                    "append_fields": APPEND_FIELDS, "append_nulls_as_none": APPEND_NULLS_AS_NONE,
                    "blacklist_file": BLACKLIST_FILE, "text_col": TEXT_COL, "cleaned_col": CLEANED_COL,
                    "dataset_folder": TS})
tracker.log_metrics({**row_counts, **stats})
tracker.log_artifact(files["metadata"])
tracker.log_artifact(files["vocabulary"])
tracker.end()

raw data rows      :      814,963
rows dropped       :        2,867
cleaned data rows  :      812,096
unique tokens      :       33,182
unique bigrams     :      473,283
tokens + bigrams   :      506,465
